# DJIWaypointTool

This notebook provides an interactive ipyleaflet map for:

- Planning a lawnmower flight path (waypoints) around a draggable mission-center pin
- Importing and visualizing multiple KML or KMZ trail files, each as an independent layer with its own color

Run the notebook top to bottom, but the final cell is designed to be safe to re-run.

In [1]:
import sys
print(sys.version)


3.12.10 (v3.12.10:0cc81280367, Apr  8 2025, 08:47:00) [Clang 13.0.0 (clang-1300.0.29.30)]


In [2]:
import ipyleaflet, ipywidgets, shapely, pyproj, lxml
print('ipyleaflet', ipyleaflet.__version__)
print('ipywidgets', ipywidgets.__version__)
print('shapely', shapely.__version__)
print('pyproj', pyproj.__version__)
print('lxml', lxml.__version__)


ipyleaflet 0.20.0
ipywidgets 8.1.8
shapely 2.1.2
pyproj 3.7.2
lxml 6.0.2


In [3]:
# === Integrated UI: Waypoint Planner (Lawnmower) + Trail Import (KML/KMZ Layers) ===
#
# Design goals
# - One map output cell, safe to re-run without duplicated controls.
# - Waypoint Panel and Trail Panel are hidden by default.
# - Workflow for flight plan placement:
#   1) Pan/zoom/search to the general area
#   2) Open Waypoint Panel
#   3) Set parameters (sliders or direct entry)
#   4) Click "Place Flight Plan"
#   5) Click the map once to place the plan using the LOWER-LEFT corner as origin (0,0,0)
# - Flight path is always SOLID BLUE.
# - Waypoints are hidden by default (optional checkbox to show them).

import io
import json
import math
import zipfile
from datetime import datetime

import ipywidgets as widgets
from ipywidgets import Layout
from IPython.display import clear_output

from lxml import etree
from pyproj import CRS, Transformer

from ipyleaflet import (
    Map, basemaps, basemap_to_tiles,
    SearchControl, WidgetControl,
    LayerGroup, Polyline, CircleMarker, Popup, Marker
)

# -----------------------------
# Hard reset (prevents duplicated controls across re-runs)
# -----------------------------
clear_output(wait=True)

try:
    if "m" in globals() and hasattr(m, "close"):
        m.close()
except Exception:
    pass

# -----------------------------
# Map
# -----------------------------
default_center = (8.614418, -77.857103)
m = Map(
    center=default_center,
    zoom=13,
    scroll_wheel_zoom=True,
    basemap=basemap_to_tiles(basemaps.OpenStreetMap.Mapnik),
    layout=Layout(width="100%", height="700px")
)

m.add_control(SearchControl(
    position="topleft",
    url="https://nominatim.openstreetmap.org/search?format=json&q={s}",
    zoom=15
))

# -----------------------------
# Flight overlays (path + optional waypoint markers + origin marker)
# -----------------------------
flight_path_layer = LayerGroup()
flight_waypoints_layer = LayerGroup()
flight_origin_layer = LayerGroup()

m.add_layer(flight_path_layer)
# Waypoints layer is NOT added by default (performance); added only when checkbox is enabled.
m.add_layer(flight_origin_layer)

# -----------------------------
# Helpers: choose a local projected CRS for meter-accurate offsets
# -----------------------------
def _utm_epsg_for(lat, lon):
    zone = int((lon + 180) // 6) + 1
    return (32600 + zone) if lat >= 0 else (32700 + zone)

def _local_transformers(lat, lon):
    epsg = _utm_epsg_for(lat, lon)
    crs_wgs84 = CRS.from_epsg(4326)
    crs_utm = CRS.from_epsg(epsg)
    to_xy = Transformer.from_crs(crs_wgs84, crs_utm, always_xy=True)
    to_ll = Transformer.from_crs(crs_utm, crs_wgs84, always_xy=True)
    return to_xy, to_ll

def _rotate_xy(x, y, bearing_deg):
    # Bearing convention:
    # - 0 deg  => sweep direction points East (+x)
    # - 90 deg => sweep direction points North (+y)
    th = math.radians(bearing_deg)
    xr = x * math.cos(th) - y * math.sin(th)
    yr = x * math.sin(th) + y * math.cos(th)
    return xr, yr

def _build_lawnmower_offsets(width_m, height_m, spacing_m):
    # Origin is lower-left corner in local (unrotated) coordinates:
    # x increases to the right (east), y increases up (north).
    #
    # We generate horizontal passes along +x, stepping +y by spacing,
    # alternating direction each row.
    if spacing_m <= 0:
        raise ValueError("Spacing must be > 0.")
    if width_m <= 0 or height_m <= 0:
        raise ValueError("Width and Height must be > 0.")

    rows = int(math.floor(height_m / spacing_m)) + 1
    y_vals = [min(i * spacing_m, height_m) for i in range(rows)]

    pts = []
    for i, y in enumerate(y_vals):
        if i % 2 == 0:
            pts.append((0.0, y))
            pts.append((width_m, y))
        else:
            pts.append((width_m, y))
            pts.append((0.0, y))

    # Ensure we end exactly on the boundary line for a clean rectangle edge
    if y_vals and y_vals[-1] != height_m:
        last_y = height_m
        if len(y_vals) % 2 == 1:
            pts.append((width_m, last_y))
        else:
            pts.append((0.0, last_y))
    return pts

def _offsets_to_latlon(origin_lat, origin_lon, offsets_xy, bearing_deg):
    to_xy, to_ll = _local_transformers(origin_lat, origin_lon)
    ox, oy = to_xy.transform(origin_lon, origin_lat)  # always_xy=True => (lon,lat)
    latlons = []
    for x, y in offsets_xy:
        xr, yr = _rotate_xy(x, y, bearing_deg)
        lon, lat = to_ll.transform(ox + xr, oy + yr)
        latlons.append((lat, lon))
    return latlons

def _clear_flight_path():
    for lyr in list(flight_path_layer.layers):
        try:
            flight_path_layer.remove_layer(lyr)
        except Exception:
            pass
    for lyr in list(flight_origin_layer.layers):
        try:
            flight_origin_layer.remove_layer(lyr)
        except Exception:
            pass
    for lyr in list(flight_waypoints_layer.layers):
        try:
            flight_waypoints_layer.remove_layer(lyr)
        except Exception:
            pass

def _ensure_waypoints_layer(is_enabled: bool):
    if is_enabled:
        if flight_waypoints_layer not in list(m.layers):
            m.add_layer(flight_waypoints_layer)
    else:
        try:
            m.remove_layer(flight_waypoints_layer)
        except Exception:
            pass

# -----------------------------
# Waypoint Planner UI
# -----------------------------
toggle_wp_panel_btn = widgets.ToggleButton(
    value=False,  # hidden by default
    description="Show Waypoint Panel",
    tooltip="Show/hide waypoint planner controls"
)

mission_name = widgets.Text(value="Test_Lawnmower", description="Name", layout=Layout(width="360px"))

alt_slider = widgets.IntSlider(value=70, min=10, max=120, step=1, description="Alt m", continuous_update=False)
alt_text = widgets.IntText(value=70, layout=Layout(width="80px"))
widgets.jslink((alt_slider, "value"), (alt_text, "value"))

speed_slider = widgets.FloatSlider(value=6.0, min=1.0, max=15.0, step=0.1, description="Speed", continuous_update=False)
speed_text = widgets.FloatText(value=6.0, layout=Layout(width="80px"))
widgets.jslink((speed_slider, "value"), (speed_text, "value"))

width_slider = widgets.IntSlider(value=1200, min=50, max=4000, step=10, description="Width m", continuous_update=False)
width_text = widgets.IntText(value=1200, layout=Layout(width="80px"))
widgets.jslink((width_slider, "value"), (width_text, "value"))

height_slider = widgets.IntSlider(value=150, min=50, max=4000, step=10, description="Height m", continuous_update=False)
height_text = widgets.IntText(value=150, layout=Layout(width="80px"))
widgets.jslink((height_slider, "value"), (height_text, "value"))

spacing_slider = widgets.IntSlider(value=50, min=5, max=500, step=5, description="Spacing", continuous_update=False)
spacing_text = widgets.IntText(value=50, layout=Layout(width="80px"))
widgets.jslink((spacing_slider, "value"), (spacing_text, "value"))

bearing_slider = widgets.IntSlider(value=270, min=0, max=359, step=1, description="Bearing", continuous_update=False)
bearing_text = widgets.IntText(value=270, layout=Layout(width="80px"))
widgets.jslink((bearing_slider, "value"), (bearing_text, "value"))

show_waypoints_cb = widgets.Checkbox(value=False, description="Show waypoints (slower)", indent=False)

place_btn = widgets.Button(description="Place Flight Plan", button_style="primary")
clear_btn = widgets.Button(description="Clear Flight Path", button_style="")

wp_status = widgets.HTML(value="<b>Flight Path:</b> set parameters, then click <b>Place Flight Plan</b>.")

def _row(slider, txt):
    return widgets.HBox([slider, txt], layout=Layout(width="380px"))

wp_controls_box = widgets.VBox(
    [
        widgets.HTML("<b>Waypoint Planner</b>"),
        widgets.HTML("Workflow: configure parameters, click <b>Place Flight Plan</b>, then click the map to place the <b>lower-left</b> corner."),
        mission_name,
        _row(alt_slider, alt_text),
        _row(speed_slider, speed_text),
        _row(width_slider, width_text),
        _row(height_slider, height_text),
        _row(spacing_slider, spacing_text),
        _row(bearing_slider, bearing_text),
        show_waypoints_cb,
        widgets.HBox([place_btn, clear_btn]),
        wp_status,
    ],
    layout=Layout(width="420px")
)

wp_panel_container = widgets.VBox(
    [widgets.HBox([toggle_wp_panel_btn]), wp_controls_box],
    layout=Layout(width="440px")
)

def _update_wp_panel_visibility(change=None):
    if toggle_wp_panel_btn.value:
        toggle_wp_panel_btn.description = "Hide Waypoint Panel"
        wp_controls_box.layout.display = "flex"
        wp_panel_container.layout.width = "440px"
    else:
        toggle_wp_panel_btn.description = "Show Waypoint Panel"
        wp_controls_box.layout.display = "none"
        wp_panel_container.layout.width = "180px"

toggle_wp_panel_btn.observe(_update_wp_panel_visibility, names=["value"])
_update_wp_panel_visibility()

# Remove previous wp control (in case of partial rerun in same kernel)
if "wp_panel_control" in globals():
    try:
        m.remove_control(wp_panel_control)
    except Exception:
        pass

wp_panel_control = WidgetControl(widget=wp_panel_container, position="topright")
m.add_control(wp_panel_control)

# Placement state:
# - when True, the next click on the map places the origin and builds the path
awaiting_origin_click = {"active": False}

def _render_flight_plan(origin_lat, origin_lon):
    _clear_flight_path()

    width_m = float(width_slider.value)
    height_m = float(height_slider.value)
    spacing_m = float(spacing_slider.value)
    bearing_deg = float(bearing_slider.value)

    offsets = _build_lawnmower_offsets(width_m, height_m, spacing_m)
    latlons = _offsets_to_latlon(origin_lat, origin_lon, offsets, bearing_deg)

    # Draw flight path (SOLID BLUE)
    path = Polyline(locations=latlons, weight=4, color="#0066FF")  # solid blue
    flight_path_layer.add_layer(path)

    # Origin marker (lower-left)
    origin_marker = CircleMarker(
        location=(origin_lat, origin_lon),
        radius=8,
        color="#0066FF",
        fill_color="#0066FF",
        fill_opacity=0.9
    )
    origin_marker.popup = Popup(
        child=widgets.HTML(value=f"<b>Origin (0,0,0)</b><br>{origin_lat:.6f}, {origin_lon:.6f}"),
        auto_close=False,
        close_button=True
    )
    flight_origin_layer.add_layer(origin_marker)

    # Waypoints (optional)
    _ensure_waypoints_layer(show_waypoints_cb.value)
    if show_waypoints_cb.value:
        for idx, (lat, lon) in enumerate(latlons):
            mk = CircleMarker(
                location=(lat, lon),
                radius=4,
                color="#0066FF",
                fill_color="#0066FF",
                fill_opacity=0.9
            )
            mk.popup = Popup(child=widgets.HTML(value=f"<b>WP {idx}</b><br>{lat:.6f}, {lon:.6f}"))
            flight_waypoints_layer.add_layer(mk)
    else:
        # Make sure the layer is empty if disabled
        for lyr in list(flight_waypoints_layer.layers):
            try:
                flight_waypoints_layer.remove_layer(lyr)
            except Exception:
                pass

    wp_status.value = (
        f"<b>Flight Path:</b> {len(latlons)} point(s), origin {origin_lat:.6f}, {origin_lon:.6f}, "
        f"alt {alt_slider.value} m, speed {speed_slider.value:.1f} m/s"
    )

def _on_place_clicked(b=None):
    awaiting_origin_click["active"] = True
    wp_status.value = "<b>Flight Path:</b> click the map to place the <b>lower-left</b> corner (origin 0,0,0)."

def _on_clear_clicked(b=None):
    awaiting_origin_click["active"] = False
    _clear_flight_path()
    wp_status.value = "<b>Flight Path:</b> cleared."

def _on_show_waypoints_change(change):
    if change.get("name") != "value":
        return
    _ensure_waypoints_layer(change["new"])
    # Do not regenerate automatically; just hide/show current markers if any.
    if not change["new"]:
        for lyr in list(flight_waypoints_layer.layers):
            try:
                flight_waypoints_layer.remove_layer(lyr)
            except Exception:
                pass

place_btn.on_click(_on_place_clicked)
clear_btn.on_click(_on_clear_clicked)
show_waypoints_cb.observe(_on_show_waypoints_change, names=["value"])

# Map click handler: ONLY place when we're in "awaiting origin click" mode.
def _on_map_interaction(**kwargs):
    if kwargs.get("type") != "click":
        return
    if not awaiting_origin_click["active"]:
        return

    lat, lon = kwargs.get("coordinates", (None, None))
    if lat is None:
        return

    awaiting_origin_click["active"] = False
    _render_flight_plan(lat, lon)

m.on_interaction(_on_map_interaction)

# -----------------------------
# Trail Import (KML/KMZ → Multi-Layer, Color Picker, Save/Load, Collapsible Panel)
# -----------------------------
# This is the user's previously working trail system, adapted to:
# - Default hidden panel
# - Safe re-run without control duplication
# - Trails remain separate from flight overlays

# ---------- Storage for trail layers ----------
trail_layers = getattr(m, "_trail_layers", [])
m._trail_layers = trail_layers

# ---------- UI widgets ----------
trail_upload = widgets.FileUpload(accept=".kml,.kmz", multiple=False, description="Upload KML/KMZ")

trail_layer_name = widgets.Text(
    value="Trail Layer",
    description="Layer name",
    placeholder="e.g., Blue Trail System",
    layout=widgets.Layout(width="340px")
)

trail_color = widgets.ColorPicker(
    concise=False,
    description="Color",
    value="#7B3FF2",
    layout=widgets.Layout(width="340px")
)

add_layer_btn = widgets.Button(description="Add Layer from Upload", button_style="success")
clear_all_btn = widgets.Button(description="Remove All Trail Layers", button_style="")
save_btn = widgets.Button(description="Save Trails", button_style="")
load_btn = widgets.Button(description="Load Trails", button_style="")

trail_status = widgets.HTML(value="<b>Trails:</b> no layers loaded")

toggle_trail_panel_btn = widgets.ToggleButton(
    value=False,  # hidden by default
    description="Show Trail Panel",
    tooltip="Hide/show trail import controls"
)

layers_box_title = widgets.HTML("<b>Loaded Trail Layers</b>")
layers_box = widgets.VBox([])

trail_controls_box = widgets.VBox(
    [
        widgets.HTML("<b>Trail Import</b>"),
        widgets.HTML("Workflow: Upload → name → choose color → Add Layer. Repeat per trail system."),
        trail_upload,
        trail_layer_name,
        trail_color,
        add_layer_btn,
        widgets.HBox([save_btn, load_btn]),
        widgets.HBox([clear_all_btn]),
        trail_status,
        layers_box_title,
        layers_box
    ],
    layout=widgets.Layout(width="380px")
)

trail_panel_container = widgets.VBox(
    [widgets.HBox([toggle_trail_panel_btn]), trail_controls_box],
    layout=widgets.Layout(width="400px")
)

# Remove previous control if re-running
if "trail_panel_control" in globals():
    try:
        m.remove_control(trail_panel_control)
    except Exception:
        pass

trail_panel_control = WidgetControl(widget=trail_panel_container, position="bottomright")
m.add_control(trail_panel_control)

def _update_trail_panel_visibility(change=None):
    if toggle_trail_panel_btn.value:
        toggle_trail_panel_btn.description = "Hide Trail Panel"
        trail_controls_box.layout.display = "flex"
        trail_panel_container.layout.width = "400px"
    else:
        toggle_trail_panel_btn.description = "Show Trail Panel"
        trail_controls_box.layout.display = "none"
        trail_panel_container.layout.width = "180px"

toggle_trail_panel_btn.observe(_update_trail_panel_visibility, names=["value"])
_update_trail_panel_visibility()

# ---------- KML parsing helpers ----------
def _extract_kml_bytes(file_name, raw_bytes):
    lower = (file_name or "").lower()
    if lower.endswith(".kmz"):
        with zipfile.ZipFile(io.BytesIO(raw_bytes), "r") as zf:
            kml_files = [n for n in zf.namelist() if n.lower().endswith(".kml")]
            if not kml_files:
                raise ValueError("KMZ contains no .kml file.")
            return zf.read(kml_files[0])
    if lower.endswith(".kml"):
        return raw_bytes
    raise ValueError("Unsupported file type.")

def _ns(root):
    nsmap = getattr(root, "nsmap", {}) or {}
    default_ns = nsmap.get(None, "http://www.opengis.net/kml/2.2")
    return {"k": default_ns}

def _parse_point_coords(text):
    first = text.strip().split()[0]
    parts = first.split(",")
    if len(parts) < 2:
        return None
    lon = float(parts[0])
    lat = float(parts[1])
    return (lat, lon)

def _parse_linestring_coords(text):
    pts = []
    for tup in text.strip().split():
        parts = tup.split(",")
        if len(parts) < 2:
            continue
        lon = float(parts[0])
        lat = float(parts[1])
        pts.append((lat, lon))
    return pts if len(pts) >= 2 else None

def _parse_kml(kml_bytes):
    root = etree.fromstring(kml_bytes)
    ns = _ns(root)
    placemarks = root.xpath(".//k:Placemark", namespaces=ns) or root.xpath(".//Placemark")

    points = []
    lines = []

    for pm in placemarks:
        name = "Placemark"
        name_els = pm.xpath("./k:name", namespaces=ns) or pm.xpath("./name")
        if name_els and name_els[0].text:
            name = name_els[0].text.strip()

        pcoords = pm.xpath(".//k:Point/k:coordinates", namespaces=ns) or pm.xpath(".//Point/coordinates")
        if pcoords and pcoords[0].text:
            latlon = _parse_point_coords(pcoords[0].text)
            if latlon:
                points.append({"name": name, "lat": latlon[0], "lon": latlon[1]})

        lcoords = pm.xpath(".//k:LineString/k:coordinates", namespaces=ns) or pm.xpath(".//LineString/coordinates")
        if lcoords and lcoords[0].text:
            pts = _parse_linestring_coords(lcoords[0].text)
            if pts:
                lines.append({"name": name, "coords": pts})

    return points, lines

def _read_upload_payload(uploader):
    val = uploader.value

    if isinstance(val, dict) and val:
        name = next(iter(val.keys()))
        raw = val[name]["content"]
        return name, raw

    if isinstance(val, (list, tuple)) and len(val) > 0:
        payload = val[0]
        return payload.get("name", "uploaded"), payload.get("content")

    raise ValueError("Empty upload payload.")

def _fit_bounds(latlons):
    if not latlons:
        return
    lats = [p[0] for p in latlons]
    lons = [p[1] for p in latlons]
    m.fit_bounds([[min(lats), min(lons)], [max(lats), max(lons)]])

def _make_layer_group(points, lines, color_hex, show_points=True):
    lg = LayerGroup()
    all_coords = []

    for ln in lines:
        coords = ln["coords"]
        all_coords.extend(coords)
        lg.add_layer(Polyline(locations=coords, weight=3, color=color_hex))

    if show_points:
        for p in points:
            lat, lon = p["lat"], p["lon"]
            all_coords.append((lat, lon))
            marker = CircleMarker(
                location=(lat, lon),
                radius=5,
                color=color_hex,
                fill_color=color_hex,
                fill_opacity=0.9
            )
            marker.popup = Popup(
                child=widgets.HTML(value=f"<b>{p['name']}</b><br>{lat:.6f}, {lon:.6f}"),
                auto_close=False,
                close_button=True
            )
            lg.add_layer(marker)

    return lg, all_coords

def _refresh_layers_box():
    rows = []
    for idx, meta in enumerate(m._trail_layers):
        name = meta["name"]
        color = meta["color"]

        visible_cb = widgets.Checkbox(value=True, description=f"{name} ({color})", indent=False)
        remove_btn = widgets.Button(description="Remove", button_style="", layout=widgets.Layout(width="90px"))

        visible_cb.value = meta["layer"] in list(m.layers)

        def _make_toggle_handler(layer_meta):
            def _handler(change):
                if change["name"] != "value":
                    return
                if change["new"]:
                    if layer_meta["layer"] not in list(m.layers):
                        m.add_layer(layer_meta["layer"])
                else:
                    try:
                        m.remove_layer(layer_meta["layer"])
                    except Exception:
                        pass
            return _handler

        def _make_remove_handler(layer_meta):
            def _handler(b=None):
                try:
                    m.remove_layer(layer_meta["layer"])
                except Exception:
                    pass
                m._trail_layers = [x for x in m._trail_layers if x["id"] != layer_meta["id"]]
                trail_status.value = f"<b>Trails:</b> loaded {len(m._trail_layers)} layer(s)."
                _refresh_layers_box()
            return _handler

        visible_cb.observe(_make_toggle_handler(meta), names=["value"])
        remove_btn.on_click(_make_remove_handler(meta))

        rows.append(widgets.HBox([visible_cb, remove_btn]))

    layers_box.children = rows
    trail_layers[:] = m._trail_layers

def _add_layer_from_current_upload():
    if not trail_upload.value:
        raise ValueError("No upload selected.")
    file_name, raw_bytes = _read_upload_payload(trail_upload)
    kml_bytes = _extract_kml_bytes(file_name, raw_bytes)
    points, lines = _parse_kml(kml_bytes)

    if not points and not lines:
        raise ValueError("No supported Placemarks found (Point or LineString).")

    layer_name = (trail_layer_name.value or "").strip() or file_name
    color_hex = trail_color.value

    lg, all_coords = _make_layer_group(points, lines, color_hex, show_points=True)

    m.add_layer(lg)

    layer_id = datetime.utcnow().strftime("%Y%m%d%H%M%S%f")
    meta = {
        "id": layer_id,
        "name": layer_name,
        "color": color_hex,
        "points": points,
        "lines": lines,
        "layer": lg
    }
    m._trail_layers.append(meta)
    trail_status.value = f"<b>Trails:</b> loaded {len(m._trail_layers)} layer(s)."

    if all_coords:
        _fit_bounds(all_coords)

    _refresh_layers_box()

def _remove_all_trail_layers():
    for meta in list(m._trail_layers):
        try:
            m.remove_layer(meta["layer"])
        except Exception:
            pass
    m._trail_layers = []
    trail_status.value = "<b>Trails:</b> no layers loaded"
    _refresh_layers_box()

def _save_trails_to_json(path="saved_trails.json"):
    payload = []
    for meta in m._trail_layers:
        payload.append({
            "id": meta["id"],
            "name": meta["name"],
            "color": meta["color"],
            "points": meta["points"],
            "lines": meta["lines"],
        })
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    trail_status.value = f"<b>Trails:</b> saved {len(payload)} layer(s) to {path}"

def _load_trails_from_json(path="saved_trails.json"):
    with open(path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    _remove_all_trail_layers()

    for item in payload:
        lg, _ = _make_layer_group(
            item.get("points", []),
            item.get("lines", []),
            item.get("color", "#7B3FF2"),
            show_points=True
        )
        m.add_layer(lg)
        m._trail_layers.append({
            "id": item.get("id", datetime.utcnow().strftime("%Y%m%d%H%M%S%f")),
            "name": item.get("name", "Trail Layer"),
            "color": item.get("color", "#7B3FF2"),
            "points": item.get("points", []),
            "lines": item.get("lines", []),
            "layer": lg
        })

    trail_status.value = f"<b>Trails:</b> loaded {len(m._trail_layers)} layer(s) from {path}"
    _refresh_layers_box()

def _on_add_clicked(b=None):
    try:
        _add_layer_from_current_upload()
    except Exception as e:
        trail_status.value = f"<b>Trails:</b> Error: {str(e)}"

def _on_clear_all_clicked(b=None):
    _remove_all_trail_layers()

def _on_save_clicked(b=None):
    try:
        _save_trails_to_json("saved_trails.json")
    except Exception as e:
        trail_status.value = f"<b>Trails:</b> Save error: {str(e)}"

def _on_load_clicked(b=None):
    try:
        _load_trails_from_json("saved_trails.json")
    except Exception as e:
        trail_status.value = f"<b>Trails:</b> Load error: {str(e)}"

add_layer_btn.on_click(_on_add_clicked)
clear_all_btn.on_click(_on_clear_all_clicked)
save_btn.on_click(_on_save_clicked)
load_btn.on_click(_on_load_clicked)

_refresh_layers_box()

m


Map(center=[8.614418, -77.857103], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title',…

## Next steps

- Add an exporter that emits DJI Fly compatible KMZ missions for Air 3.
- Optionally add a "Deploy" helper for copying the KMZ to the RC 2 via OpenMTP or ADB.
